In [ ]:
#THIS NOTEBOOK IS THE BASELINE PREDICTION MODEL FOR MELTING TEMPERATURE (Tm)

from chemprop_utils import load_and_clean_data, build_splits, build_and_train_mpnn, predict_test_set    #IMPORTS REQUIRED FUNCTIONS FROM THE FUNCTION SCRIPT
from pathlib import Path                    #IMPORTS THE FILE PATH HANDLING LIBRARY
import pandas as pd                         #IMPORTS THE DATA FRAME HANDLING LIBRARY
from lightning import pytorch as pl         #IMPORTS PYTORCH LIGHTNING, USED FOR SEEDING AND TRAINING

pl.seed_everything(0, workers=True)         #FIXES THE RANDOM SEED FOR REPRODUCIBILITY (ALSO SEEDS DATALOADER WORKER PROCESSES)


# SPLIT: SCAFFOLD_BALANCED SPLIT WILL BE USED AS IT OUTPERFORMED RANDOM SPLIT RUNS (RMSE ~2.35 VS RMSE ~4)
# RMSE WILL BE USED AS TRAINING OBJECTIVE. SENSITIVITY RSEULTS WERE INCONCLUSIVE WITH REGARDS TO AN OUTRIGHT OPTIMAL TRAINING OBJECTIVE (MINOR DIFFERENCES ACROSS RUNS), BUT RMSE IS CONSIDERED STANDARD
# DROPOUT OF 0.4 WILL BE USED AS IT RESULTED IN THE LOWEST MAE OF ALL TESTED VALUES
# NO FOLDING WILL BE DONE (SINGLE 80/10/10 SPLIT) TO INCREASE RUN SPEED

input_path = Path("Tmp_DB.csv")                          #PATH TO Tm DATA — ADJUST IF FILE SITS IN A SUBFOLDER
smiles_column = "smiles"                                 #DEFINES THE SMILES COLUMN — CONFIRM THIS MATCHES YOUR Tm CSV HEADER
target_columns = ["Tm [K]"]                              #DEFINES THE MELTING TEMPERATURE COLUMN AS THE TARGET — CONFIRM EXACT COLUMN NAME

df_clean, all_data, mols = load_and_clean_data(input_path, smiles_column, target_columns)       #RUNS THE DATA CLEANING AND LOADING FUNCTION
splits = build_splits(all_data, mols, split_types=("scaffold_balanced",))                       #CALLS THE SPLITTING FUNCTION USING SCAFFOLD SPLITTING ONLY

result_baseline = build_and_train_mpnn(                  #TRAINS THE MPNN USING THE LOCKED CONFIGURATION
    splits["scaffold_balanced"]["train"],
    splits["scaffold_balanced"]["val"],
    splits["scaffold_balanced"]["test"],
    dropout=0.4,
    run_name="Tm_OFFICIAL_BASELINE",
)

test_metrics = result_baseline["test_results"][0]        #EXTRACTS THE ERROR METRICS OF THE TEST RESULTS
print(f"OFFICIAL Tm BASELINE — RMSE: {test_metrics['test/rmse']:.3f}, MAE: {test_metrics['test/mae']:.3f}")

baseline_predictions = predict_test_set(result_baseline, run_name="Tm_OFFICIAL_BASELINE")       #CALLS THE PREDICTION FUNCTION
baseline_predictions.to_csv("Tm_baseline_predictions.csv", index=False)                         #SAVES THE BASELINE PREDICTIONS

#NOTE: RESULTS AND SPLITS MUST BE SAVED TO BE USED IN SUBSEQUENT NOTEBOOKS (E.G. MULTI-TASK)
import pickle                                            #IMPORT LIBRARY FOR DATA SAVING
with open("Tm_baseline_splits.pkl", "wb") as f:          #OPENS A NEW FILE FOR WRITING
    pickle.dump(splits, f)                               #SAVE THE SPLITS BY WRITING TO THE FILE

print("Tm baseline locked in. Predictions and splits saved for downstream notebooks.")